# Recomendador de Sinopses - emblenddings reais com textos pequenos

Antes de carregar milhares de animes, vamos testar o processo com apenas algumas frases curtas. Isso nos permite verificar separadamente:

1. se conseguimos carregar um modelo de embeddings;
2. se textos semanticamente parecidos ficam próximos;
3. qual é o formato real dos vetores;
4. como o NumPy calcula a similaridade;
5. como visualizar um vetor com centenas de dimensões.

> Esta etapa já usa uma IA para gerar embeddings, mas ainda não usa um dataset grande, banco vetorial ou chatbot.
Antes de aplicarmos nossa técnica em um dataset gigante com milhares de sinopses de animes, precisamos garantir que o núcleo do nosso sistema funciona com textos curtos e reais. Para que o recomendador de animes "entenda" o que o usuário quer, usaremos a Similaridade do Cosseno. Essa métrica não compara as palavras exatas, mas sim a direção dos vetores no espaço multidimensional gerado pela IA. 


## Frases em emblenddings

In [8]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

frases = [
    "Uma aventura de fantasia em um mundo mágico.",
    "Uma jornada fantástica por um reino cheio de magia.",
    "Uma comédia escolar sobre amizade e música.",
    "Um grupo de heróis enfrenta monstros em uma longa jornada.",
]

embeddings = modelo.encode(frases)
 
print("Formatos dos embenddiings: ", embeddings.shape)

print("Primeiros 12 valores do primeiro embedding:")
print(embeddings[0][:12])

print("Tipo da matriz:", type(embeddings))
print("Tipo de um valor:", type(embeddings[0][0]))



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2366.51it/s]


Formatos dos embenddiings:  (4, 384)
Primeiros 12 valores do primeiro embedding:
[ 0.16176815  0.09984124 -0.26699555  0.18294364 -0.6823191  -0.4359224
  0.41151875 -0.24203688  0.04108814  0.17685072  0.47948417 -0.12003288]
Tipo da matriz: <class 'numpy.ndarray'>
Tipo de um valor: <class 'numpy.float32'>


#### O que esse codigo faz? 

##### Escopo inicial
O código começa preparando o terreno e executando a transformação matemática antes de exibir qualquer resultado na tela.

-  `modelo = SentenceTransformer(...)` :  Esta instrução instancia o modelo de linguagem pré-treinado na memória. O modelo especificado (paraphrase-multilingual-MiniLM-L12-v2) utiliza uma arquitetura baseada em Transformers, otimizada para processar múltiplos idiomas (incluindo o português) com alta eficiência computacional.

- `frases = [...]`:  É uma lista comum do Python contendo quatro textos com contextos diferentes.


- `embeddings = modelo.encode(frases)`: Representa a etapa de processamento principal. O método .encode() recebe as sentenças, realiza a tokenização e aplica os cálculos da rede neural para mapear os textos em um espaço vetorial. O resultado desse mapeamento é armazenado na variável embeddings.

Observação: O método `.encode()` preserva estritamente o mapeamento 1:1 da lista de entrada, vinculando o índice do texto original ao mesmo índice numérico na matriz gerada. A variável `embeddings` não é um vetor linear, mas uma matriz bidimensional (ex: 4 linhas/sentenças por 384 colunas/dimensões). Ao utilizar um único indexador estrutural, como `embeddings[1]`, a linguagem extrai a linha completa correspondente, entregando o vetor denso integral (todas as 384 coordenadas conjuntas) necessário para o cálculo direcional. O acesso a um ponto escalar isolado exigiria obrigatoriamente a indexação da segunda dimensão da matriz, como `embeddings[1][0]`. 
Ou seja embedding( Número de frases, Número de pontos do vetor) = (4, 384) isso significa que ela tem 4 linhas (frases) e 384 colunas (pontos do vetor).

##### Análise Estrutural (Função de cada print)

- `print("Formatos dos embenddiings: ", embeddings.shape)`: Extrai as dimensões exatas da matriz numérica gerada. O atributo .shape retornará a tupla (4, 384). O valor 4 corresponde ao tamanho do lote (batch size), refletindo as quatro frases de entrada. O valor 384 indica a dimensionalidade da representação vetorial. Isso confirma tecnicamente que cada frase foi convertida em um vetor de 384 coordenadas contínuas.

- `print(embeddings[0][:12])`: Acessa e exibe uma amostra transversal dos dados numéricos extraídos. Por meio do fatiamento em Python ([0][:12]), o comando isola o primeiro vetor (referente à primeira frase) e exibe apenas seus 12 primeiros escalares. O objetivo é inspecionar empiricamente as magnitudes numéricas (valores positivos e negativos) que quantificam as características semânticas do texto.

- `print("Tipo da matriz:", type(embeddings))`: Identifica a classe da estrutura de dados que armazena os embeddings. O retorno será <class 'numpy.ndarray'>. Isso demonstra que o método .encode() não gera uma lista nativa do Python, mas sim uma matriz multidimensional da biblioteca NumPy. Essa estrutura é fundamental para permitir operações de álgebra linear e processamento paralelo (essenciais para cálculos de similaridade, como a distância de cosseno).

- `print("Tipo de um valor:", type(embeddings[0][0]))`: Verifica a tipagem e a precisão de ponto flutuante de um único elemento dentro da matriz. O retorno será <class 'numpy.float32'>. Esta validação é crucial sob a ótica de engenharia de Machine Learning. Ela indica que a biblioteca alocou dados em precisão de 32 bits (em vez dos 64 bits padrão do Python), reduzindo pela metade o consumo de memória RAM. Este formato é o padrão exigido para aceleração em hardware (GPUs) e para indexação eficiente em bancos de dados vetoriais.


## Calculando a similaridade do cosseno

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer

def similaridadeCosseno(vetorA, vetorB):
    normA = np.linalg.norm(vetorA)
    normB = np.linalg.norm(vetorB)
    
    if normA == 0 or normB == 0:
        return 0.0
    
    produtoescalar = np.dot(vetorA,vetorB)
    return produtoescalar / (normA * normB)

modelo = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

frases = [
    "Uma aventura de fantasia em um mundo mágico.",
    "Uma jornada fantástica por um reino cheio de magia.",
    "Uma comédia escolar sobre amizade e música.",
    "Um grupo de heróis enfrenta monstros em uma longa jornada.",
]

embeddings = modelo.encode(frases)

Similaridade01 = similaridadeCosseno(
    embeddings[0],
    embeddings[1]
)

Similaridade02 = similaridadeCosseno(
    embeddings[0],
    embeddings[2]
)

Similaridade03 = similaridadeCosseno(
    embeddings[0],
    embeddings[3]
)

print("Frase 1:", frases[0])
print("Frase 2:", frases[1])
print("Similaridade:", Similaridade01)
print()

print("Frase 1:", frases[0])
print("Frase 3:", frases[2])
print("Similaridade:", Similaridade02)
print()

print("Frase 1:", frases[0])
print("Frase 4:", frases[3])
print("Similaridade:", Similaridade03)
print()



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2431.25it/s]


Frase 1: Uma aventura de fantasia em um mundo mágico.
Frase 2: Uma jornada fantástica por um reino cheio de magia.
Similaridade: 0.7964472

Frase 1: Uma aventura de fantasia em um mundo mágico.
Frase 3: Uma comédia escolar sobre amizade e música.
Similaridade: 0.3136834

Frase 1: Uma aventura de fantasia em um mundo mágico.
Frase 4: Um grupo de heróis enfrenta monstros em uma longa jornada.
Similaridade: 0.4403108



#### O que esse código faz?
Este código avança para a aplicação prática dos dados, implementando a lógica de comparação para descobrir quais os textos mais parecidos entre si.   Em termos de revisão matemática, a função manual similaridadeCosseno cruza as matrizes multiplicando as coordenadas de dois vetores (produto escalar) e divide o resultado pelo comprimento de ambos (as normas), o que normaliza a pontuação e previne falhas críticas no programa, como divisões por zero.   Na prática, o que este guião faz de novo é isolar o vetor correspondente à primeira frase (embeddings[0]) e invocar repetidamente essa função para o comparar, um a um, com os vetores da segunda, terceira e quarta frases. No final, o código estrutura estes resultados com comandos de impressão, colocando os textos lado a lado com os seus respetivos coeficientes matemáticos, permitindo visualizar de forma clara qual das opções partilha uma maior afinidade semântica com a frase original.   